[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/appars/codepilot-colab/blob/main/stage5_reflection/Stage5_Reflection.ipynb)

> **Click the badge above to open this notebook in Google Colab.**
> Or go directly: https://colab.research.google.com/github/appars/codepilot-colab/blob/main/stage5_reflection/Stage5_Reflection.ipynb

# 🔄 Stage 5 — Reflection Loops
**CodePilot AI Studio | Module 4 | Homework**

---

## What You Will Learn
- What a **reflection loop** is in agentic AI
- How **Generate → Critique → Improve** works in practice
- Why self-critique produces dramatically better output
- How `max_iterations` prevents infinite loops

## The Concept
```
Student WITHOUT reflection:    Student WITH reflection:
Write answer → submit          Write answer → re-read →
                               spot mistake → fix → check again
Result: mediocre               Result: much better!
```

## The Reflection Loop
```
Generate fix
     ↓
Critique fix → NEEDS_IMPROVEMENT → Improve fix → loop back
     ↓
   APPROVED
     ↓
Return final fix
```

⏱ **Expected time: 25 minutes**

## Step 1 — Add Your Groq API Key (One Time Setup)

### Option A — Using Colab Secrets (Recommended!)
Store your key ONCE in Colab Secrets and it works in ALL notebooks automatically:
1. Click the **🔑 key icon** in the left sidebar (or go to Tools → Secrets)
2. Click **'Add new secret'**
3. Name: `GROQ_API_KEY`  (must be exactly this name)
4. Value: paste your key (looks like `gsk_xxxx...`)
5. Toggle **'Notebook access'** to ON
6. Come back and run this cell

### Option B — Paste directly (quick one-time use)
If you do not want to use Secrets, just paste your key in the code cell below.

> Get your free key at **https://console.groq.com** → API Keys → Create API Key

In [ ]:
# ── GROQ API KEY SETUP ───────────────────────────────────────
# This cell tries Colab Secrets first (recommended).
# If not found, falls back to manual paste below.

import os

# ── METHOD 1: Colab Secrets (store once, works in all notebooks)
# If you added GROQ_API_KEY in the Secrets panel, this will find it.
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    if GROQ_API_KEY:
        print("Groq API key loaded from Colab Secrets!")
        print(f"Key starts with: {GROQ_API_KEY[:8]}...")
    else:
        raise ValueError("Key not found in Secrets")
except Exception as e:
    # ── METHOD 2: Manual paste (fallback)
    # If Secrets is not set up, paste your key here:
    GROQ_API_KEY = "paste-your-groq-key-here"   # ← replace if not using Secrets
    if GROQ_API_KEY == "paste-your-groq-key-here":
        print("ERROR: Key not found in Secrets and not pasted manually.")
        print("")
        print("Option A: Add to Colab Secrets:")
        print("  1. Click the key icon in the left sidebar")
        print("  2. Add secret name: GROQ_API_KEY")
        print("  3. Paste your key as the value")
        print("  4. Enable Notebook access and re-run this cell")
        print("")
        print("Option B: Paste your key directly above (replace 'paste-your-groq-key-here')")
        print("Get your free key from: https://console.groq.com")
    else:
        print(f"Groq API key set manually. Starts with: {GROQ_API_KEY[:8]}...")

# Set as environment variable so LangChain reads it automatically
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# Final check
if len(GROQ_API_KEY) > 20:
    print("Ready to proceed!")


## Step 2 — Setup

In [ ]:
print("Installing packages...")
!pip install -q langchain-groq langchain langchain-community langchain-core langchain-text-splitters langgraph
print("Setup complete!")

In [ ]:
# ============================================================
# CodePilot AI Studio — Stage 5: Reflection Loops
# ============================================================

from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

# temperature=0.4 — slightly creative for generating fixes
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.4)

# ── THREE PROMPTS — same model, three different roles ─────────
GENERATE_PROMPT = PromptTemplate(
    input_variables=["code"],
    template="""You are a Python developer fixing bugs.
Analyse and fix this code. This is a first draft — it will be reviewed.
Provide: 1. Bug found 2. Fixed code 3. Brief explanation

Code: ```python\n{code}\n```"""
)

CRITIQUE_PROMPT = PromptTemplate(
    input_variables=["original_code", "proposed_fix"],
    template="""You are a strict senior Python engineer reviewing a fix.

Original code: ```python\n{original_code}\n```
Proposed fix: {proposed_fix}

Review:
1. IS THE FIX CORRECT? Does it solve ALL bugs? (YES/NO + reason)
2. EDGE CASES MISSED: Any inputs that still crash?
3. CODE QUALITY: Style or readability issues?
4. VERDICT: Write exactly APPROVED or NEEDS_IMPROVEMENT (then explain)

Be strict — a good fix handles ALL edge cases."""
)

IMPROVE_PROMPT = PromptTemplate(
    input_variables=["original_code", "previous_fix", "critique"],
    template="""You are a Python expert improving a fix based on review feedback.

Original code: ```python\n{original_code}\n```
Previous fix: {previous_fix}
Review feedback: {critique}

Produce an IMPROVED fix that addresses ALL feedback:
1. IMPROVED FIX: [corrected code]
2. WHAT CHANGED: [list every improvement]
3. EDGE CASES HANDLED: [confirm all cases work]"""
)

# ── THE REFLECTION LOOP ───────────────────────────────────────
def reflection_review(code: str, max_iterations: int = 2) -> dict:
    """
    Run the full Generate → Critique → Improve loop.
    Stops early when critique says APPROVED.
    Never runs more than max_iterations times (safety guard).
    """
    results = {"initial_fix": "", "critiques": [], "final_fix": "",
               "iterations": 0, "approved": False}

    # Phase 1: Generate first fix
    print("  Step 1: Generating initial fix...")
    current_fix = llm.invoke(GENERATE_PROMPT.format(code=code)).content
    results["initial_fix"] = current_fix
    print(f"  Initial fix: {len(current_fix)} chars")

    # Phase 2: Critique → Improve loop
    for i in range(max_iterations):
        print(f"\n  Step 2 (iteration {i+1}): Critiquing the fix...")
        critique = llm.invoke(CRITIQUE_PROMPT.format(
            original_code=code, proposed_fix=current_fix
        )).content
        results["critiques"].append(critique)
        results["iterations"] += 1

        # Check if approved — look for APPROVED keyword
        if "APPROVED" in critique.upper() and "NEEDS_IMPROVEMENT" not in critique.upper():
            print(f"  Critique APPROVED after {i+1} iteration(s)!")
            results["approved"] = True
            results["final_fix"] = current_fix
            break

        # Not approved — improve
        print("  Critique says NEEDS_IMPROVEMENT. Improving...")
        current_fix = llm.invoke(IMPROVE_PROMPT.format(
            original_code=code,
            previous_fix=current_fix,
            critique=critique
        )).content

    if not results["approved"]:
        print(f"  Max iterations ({max_iterations}) reached.")
        results["final_fix"] = current_fix

    return results

# ── RUN THE DEMO ──────────────────────────────────────────────
buggy_code = """
def calculate_average(numbers):
    total = 0
    for num in numbers:
        total = total + num
    return total / len(numbers)

print(calculate_average([10, 20, 30]))
print(calculate_average([]))          # ZeroDivisionError!
print(calculate_average("hello"))     # TypeError!
"""

print("=" * 60)
print("CodePilot AI Studio — Stage 5: Reflection Loops")
print("=" * 60)
print("Buggy code:", buggy_code)
print("Starting reflection loop...")

results = reflection_review(buggy_code, max_iterations=2)

print()
print("=" * 60)
print("INITIAL FIX (before reflection):")
print(results["initial_fix"][:300])
print()
print("FINAL FIX (after reflection):")
print(results["final_fix"][:400])
print()
print("=" * 60)
print(f"Iterations run : {results['iterations']}")
print(f"Approved       : {results['approved']}")
print()
print("KEY: The agent improved its OWN output — no human intervention!")

## Step 5 — Experiments

In [ ]:
# ── EXPERIMENT: Compare initial vs final fix ──────────────────
print("INITIAL FIX (first draft):")
print("-" * 40)
print(results["initial_fix"])
print()
print("FINAL FIX (after reflection):")
print("-" * 40)
print(results["final_fix"])
print()
print("Q: What specific improvements did reflection add?")
print("Q: Try max_iterations=1 — is quality lower?")

## Summary — Stage 5 Complete!

| Concept | What it means |
|---------|---------------|
| **Reflection loop** | Agent evaluates and improves its own output |
| **Generate → Critique → Improve** | The three phases |
| **Early stopping** | Exit when APPROVED — no need to keep going |
| **`max_iterations`** | Safety guard against infinite loops |

➡️ Continue with `stage6_langgraph/Stage6_LangGraph.ipynb`